# GenAI Pipeline — Testing Notebook

LLM-based screening of publications for alternative protein relevance and pillar classification.
Uses Claude with prompt caching via the Anthropic Python SDK.

### 1. Imports and Configuration

In [147]:
import duckdb
import pandas as pd
import json
import random
import time
import anthropic
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator
from typing import Literal
from pydantic import create_model

load_dotenv()

OUTPUT_DIR = Path(".")
RAW_SUBS_DIR = Path("3_research-category/raw_subsets/")

### 2. Data Inspection

In [148]:
EXCEL_PATH = Path("1_deduplication/raw_data/Funding2026_inscope.xlsx")
df_raw = pd.read_excel(EXCEL_PATH)

# All records in this file are already in-scope; require a Research category label
# AND a non-empty Abstract. The automated (LLM) pipeline should only ever see records
# it has enough text to classify — records with a category but no abstract are set
# aside for manual review instead of silently dropped.
has_category = df_raw["Research category"].notna() & (df_raw["Research category"].str.strip() != "")
has_abstract = df_raw["Abstract"].notna() & (df_raw["Abstract"].str.strip() != "")

df = df_raw[has_category & has_abstract].reset_index(drop=True)
needs_manual_review = df_raw[has_category & ~has_abstract].reset_index(drop=True)

# Clean up label inconsistencies before splitting multi-label cells.
# "Health/nutrition" and "Env/impact assessments" use "/" as part of the name, not as a
# separator, so rename them before converting remaining "/" separators to commas.
# "Consumer and market research", "Host strain development", and "Texturisation methods"
# are the same categories as "Consumer & market research", "Strain development", and
# "Texturization methods" in our category lists (PB_CATS etc.) — just worded/spelled
# differently in the raw data — so align them to the names our lists use.
# "other" -> "Other" is done here (once, at the source) rather than ad-hoc in each
# downstream function, so the displayed research_category text stays consistent
# everywhere it's shown (breakdown table, comparison, saved Excel).
df["Research category"] = (
    df["Research category"]
    .str.replace("Env/impact assessments", "Impact assessments", regex=False)
    .str.replace("Health/nutrition", "Health & nutrition", regex=False)
    .str.replace("Consumer and market research", "Consumer & market research", regex=False)
    .str.replace("Host strain development", "Strain development", regex=False)
    .str.replace("Texturisation methods", "Texturization methods", regex=False)
    .str.replace("other", "Other", regex=False)
    .str.replace("/", ",", regex=False)
)

print(f"Raw shape: {df_raw.shape}")
print(f"Filtered shape (Research category not empty, has abstract): {df.shape}")
print(f"Set aside for manual review (category present, no abstract): {needs_manual_review.shape[0]}")
print(f"\nColumns: {list(df.columns)}")
df.head()

Raw shape: (1678, 81)
Filtered shape (Research category not empty, has abstract): (916, 81)
Set aside for manual review (category present, no abstract): 734

Columns: ['Title', 'Abstract', 'Original title', 'Database', 'Total amount', 'Gov contribution', 'Currency', 'Total amount (USD)', 'Gov contribution (USD)', 'Total amount (EUR)', 'Gov & NP contribution (EUR)', 'Funding decision', 'copy to external database', 'URL for announcement', 'Identification code', 'Unnamed: 15', 'Unnamed: 16', 'Notes (external)', 'Notes (internal)', 'Project lead (PI)', 'PI department', 'PI organisation', 'PI organisation type', 'PI organisation country', 'PI organisation region', 'PI organisation state', 'PI organisation zip code', 'PI organisation congressional district', 'Collaborator names', 'Collaborator institutions', 'Multiple organisation recipients', 'Date request submitted', 'Year request submitted', 'Date award announced', 'Project start date', 'Duration of award (months)', 'Project status', 'Ann

,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035
0,Plant2Food,The new collaborative platform Plant2Food will...,NaN,airtable,200000000,200000000,DKK,28473000.0,0.0,26000000.0,...,4333333.333,4333333.333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CO2 as a sustainable raw material in our futur...,"In a new consortium, companies and university ...",NaN,airtable,100000000,100000000,DKK,27000000.0,0.0,13000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Precision Technology: Biotechnology, smart sen...",The project has three sub-goals:\n\nDeveloping...,NaN,airtable,64200000,64200000,NOK,NaN,NaN,5585400.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SEEDFOOD: Functional and palatable plant seed ...,The Foundation has awarded one of the 2021 gra...,NaN,airtable,55900000,55900000,DKK,8172831.0,0.0,7267000.0,...,1038142.857,1038142.857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,National Alternative Proteins Innovation and K...,"To secure a continued supply of safe, tasty, ...",NaN,airtable,38000000,16001352,GBP,48859450.0,18697500.0,45220000.0,...,3173601.480,3173601.480,3173601.48,3173601.48,NaN,NaN,NaN,NaN,NaN,NaN


In [149]:
# Normalise the "Plant-Based" typo variant before splitting by platform
df["Production platform"] = df["Production platform"].replace({"Plant-Based": "Plant-based"})

df_pb = df[df["Production platform"] == "Plant-based"].reset_index(drop=True)
df_f  = df[df["Production platform"] == "Fermentation"].reset_index(drop=True)
df_cm = df[df["Production platform"] == "Cultivated"].reset_index(drop=True)
df_cc = df[df["Production platform"] == "Cross-cutting"].reset_index(drop=True)

# Research category can hold multiple comma-separated labels per row; explode before counting.
def explode_categories(d):
    labels = d["Research category"].str.split(",").explode().str.strip()
    return labels.replace({"other": "Other"})

all_categories = sorted(explode_categories(df).dropna().unique())

breakdown = pd.DataFrame(
    {name: explode_categories(grp).value_counts().reindex(all_categories, fill_value=0)
     for name, grp in [("Plant-based", df_pb), ("Fermentation", df_f), ("Cultivated", df_cm), ("Cross-cutting", df_cc)]},
    index=all_categories,
)
breakdown.index.name = "research_category"
breakdown["Total"] = breakdown.sum(axis=1)
breakdown

,Plant-based,Fermentation,Cultivated,Cross-cutting,Total
research_category,,,,,
Bioprocess design,6,28,25,6,65
Cell culture media,0,0,28,1,29
Cell line development,0,0,46,1,47
Consumer & market research,45,5,4,7,61
Crop development,60,0,0,1,61
End product formulation,111,12,2,3,128
Feedstocks,1,28,0,3,32
Food safety & quality,15,3,1,4,23
Health & nutrition,48,2,0,5,55


### 3. Balanced Subset Creation

Take up to 10 records per research category. A multi-label row (e.g. "Bioprocess design, Feedstocks")
is eligible under each of its labels and is kept only once in the final set if selected more than once.

Since this grants data (unlike publications) can carry multiple category labels per record, also top up
the sample so at least 2 examples of each distinct multi-label combination are included — this specifically
tests the LLM's multi-label (true/false per category) classification.

In [155]:
RANDOM_STATE = 3
N_PER_CATEGORY = 7
MIN_PER_COMBO = 2  # ensure at least this many examples of each distinct multi-label combination

def parse_categories(series):
    """Split a comma-separated 'Research category' string into a cleaned list of labels."""
    def clean(labels):
        return ["Other" if l.strip() == "other" else l.strip() for l in labels]
    return series.str.split(",").apply(clean)

def create_balanced_sample(df, categories, n_per_category=N_PER_CATEGORY, min_per_combo=MIN_PER_COMBO, random_state=RANDOM_STATE):
    """
    Samples up to n_per_category rows per individual research category label.
    A multi-label row (e.g. "Bioprocess design, Feedstocks") is eligible under each of
    its labels, and is kept only once in the combined sample if picked more than once.
    Then tops up the sample so at least min_per_combo rows of each distinct multi-label
    combination are present, to specifically test multi-label classification.
    """
    labels = parse_categories(df["Research category"])

    samples = []
    for cat in categories:
        mask = labels.apply(lambda xs: cat in xs)
        subset = df[mask]
        available = len(subset)
        if available == 0:
            print(f"  Warning: '{cat}' — no rows found, skipping.")
            continue
        n = min(n_per_category, available)
        if n < n_per_category:
            print(f"  Warning: '{cat}' — requested {n_per_category} but only {available} available, taking all.")
        samples.append(subset.sample(n=n, random_state=random_state))

    combined = pd.concat(samples) if samples else df.iloc[0:0]
    combined = combined[~combined.index.duplicated(keep="first")]

    multi_mask = labels.apply(lambda xs: len(xs) > 1)
    multi_labels = labels[multi_mask]
    combos = multi_labels.apply(lambda xs: ", ".join(sorted(xs)))

    for combo in combos.unique():
        combo_idx = combos[combos == combo].index
        group = df.loc[combo_idx]
        already = group.index.isin(combined.index).sum()
        need = min_per_combo - already
        if need <= 0:
            continue
        remaining_pool = group[~group.index.isin(combined.index)]
        take = min(need, len(remaining_pool))
        if take < need:
            print(f"  Warning: combo '{combo}' — only {already + len(remaining_pool)} rows available, wanted {min_per_combo}.")
        if take > 0:
            combined = pd.concat([combined, remaining_pool.sample(n=take, random_state=random_state)])

    combined = combined[~combined.index.duplicated(keep="first")]
    return combined.sample(frac=1, random_state=random_state).reset_index(drop=True)

In [156]:
pb_categories = breakdown.index[breakdown["Plant-based"] > 0].tolist()
test_data_PB = create_balanced_sample(df_pb, pb_categories)
print(f"test_data_PB: {test_data_PB.shape}")
# test_data_PB[["Title", "Abstract", "Production platform", "Research category"]]

test_data_PB: (89, 81)


In [157]:
f_categories = breakdown.index[breakdown["Fermentation"] > 0].tolist()
test_data_F = create_balanced_sample(df_f, f_categories)
print(f"test_data_F: {test_data_F.shape}")
# test_data_F[["Title", "Abstract", "Production platform", "Research category"]]

test_data_F: (65, 81)


In [158]:
cm_categories = breakdown.index[breakdown["Cultivated"] > 0].tolist()
test_data_CM = create_balanced_sample(df_cm, cm_categories)
print(f"test_data_CM: {test_data_CM.shape}")
# test_data_CM[["Title", "Abstract", "Production platform", "Research category"]]

test_data_CM: (49, 81)


In [159]:
cc_categories = breakdown.index[breakdown["Cross-cutting"] > 0].tolist()
test_data_CC = create_balanced_sample(df_cc, cc_categories)
print(f"test_data_CC: {test_data_CC.shape}")
# test_data_CC[["Title", "Abstract", "Production platform", "Research category"]]

test_data_CC: (46, 81)


### 4. Save Subsets to Excel
Allows manual check of files selected. Consider whether those in the test sets are borderline cases or clear cut.

In [160]:
################################################################################################
# PLEASE CHANGE FILENAME TO THE RANDOM SEED USED IN create_balanced_sample() FOR REPRODUCIBILITY
################################################################################################
def save_subset(df, filename, output_dir=RAW_SUBS_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / filename
    df.to_excel(path, index=False)
    print(f"Saved {len(df)} records to {path}")

#save_subset(initial_test_data, "initial_test_data_rand3.xlsx")
save_subset(test_data_PB, f"rescat_test_data_PB_rand{RANDOM_STATE}_allabstracts.xlsx")
save_subset(test_data_CM, f"rescat_test_data_CM_rand{RANDOM_STATE}_allabstracts.xlsx")
save_subset(test_data_CC, f"rescat_test_data_CC_rand{RANDOM_STATE}_allabstracts.xlsx")
save_subset(test_data_F, f"rescat_test_data_F_rand{RANDOM_STATE}_allabstracts.xlsx")

Saved 89 records to 3_research-category\raw_subsets\rescat_test_data_PB_rand3_allabstracts.xlsx
Saved 49 records to 3_research-category\raw_subsets\rescat_test_data_CM_rand3_allabstracts.xlsx
Saved 46 records to 3_research-category\raw_subsets\rescat_test_data_CC_rand3_allabstracts.xlsx
Saved 65 records to 3_research-category\raw_subsets\rescat_test_data_F_rand3_allabstracts.xlsx


### 5. Load Prompt and Select Dataset

In [260]:
AP_PILLAR = "CC"  # ← CHANGE THIS: "PB", "F", "CM", "CC"

DATASETS = {
    "PB": test_data_PB,
    "F":  test_data_F,
    "CM": test_data_CM,
    "CC": test_data_CC,
}
DATASET = DATASETS[AP_PILLAR]
print(f"Pillar: {AP_PILLAR} | Dataset: {DATASET.shape[0]} records")

# ONLY USED AS REQUIRED FOR RE-RUN SPECIFIC PUBLICATIONS
#ids_to_test = ['pub.1190984789', 'pub.1196086614']
#DATASET = DATASET[DATASET["id"].isin(ids_to_test)]
#DATASET = incorrect_rescat_data
#DATASET = manual_test_data

Pillar: CC | Dataset: 46 records


In [213]:
### OPTIONAL: apply corrections to research_category from a spreadsheet of corrections

#CORRECTIONS_PATH = "2_research_category/PB_v5/PB_v5_claude-sonnet-4-6_results_only-v2-haiku-incorrect.xlsx"
#CORRECTIONS_PATH = "2_research_category/F_v5/F_v5_claude-sonnet-4-6_results_D.xlsx" 
#CORRECTIONS_PATH = "2_research_category/CM_v1/CM_v1_claude-sonnet-4-6_results.xlsx" 
#CORRECTIONS_PATH = "2_research_category/CC_v3/CC_v3_claude-sonnet-4-6_results_C.xlsx" 


#corrections = (
#    pd.read_excel(CORRECTIONS_PATH, usecols=["id", "corrected"])
#    .pipe(lambda d: d[d["corrected"].notna() & (d["corrected"].str.strip() != "")])
#    .set_index("id")["corrected"]
#    .str.strip()
#)

DATASET = DATASET.copy()
mask = DATASET["id"].isin(corrections.index)
DATASET.loc[mask, "research_category"] = DATASET.loc[mask, "id"].map(corrections)

print(f"Loaded {len(corrections)} corrections, {mask.sum()} rows updated in DATASET")

KeyError: 'id'

In [261]:
PROMPT_VERSION = "v1"  # ← CHANGE THIS to switch prompt version
PROMPT_PATH = f"3_research-category/{AP_PILLAR}_{PROMPT_VERSION}/prompt_rescat_grants_{AP_PILLAR}_{PROMPT_VERSION}.md"

In [262]:
def load_prompt(path=PROMPT_PATH):
    with open(path, "r", encoding="utf-8") as f:
        prompt_text = f.read()
    return prompt_text.strip()

system_prompt = load_prompt()
print(system_prompt)


You are an expert in alternative proteins and food science research.

Your task is to classify a grant on alternative proteins into research categories based on its title and abstract.

Before assigning categories, identify the research activities the grant will fund. Assign TRUE to every category that represents a substantive funded research activity - not merely a motivation, background context, or expected downstream application. For example:
- A grant comparing sensory properties of plant-based, mycoprotein, and cultivated meat products → End product formulation: TRUE, Consumer & market research: FALSE
- A grant studying consumer acceptance of plant-based, fermentation-derived, and cultivated meat products → Consumer & market research: TRUE
- A grant that mentions environmental benefits of alternative proteins as motivation → Impact assessments: FALSE
- A grant developing extrusion processes for both plant-based and mycoprotein products → Texturisation methods: TRUE, Ingredient opt

### 6. API Call with Prompt Caching

In [263]:
# API config

# Anthropic model options — pricing as of 2026-06-10.
# Verify at https://www.anthropic.com/pricing if costs may have changed.
# Model                  Input $/1M   Output $/1M   Context
# claude-haiku-4-5         $1.00         $5.00       200K
# claude-sonnet-4-6        $3.00        $15.00       1M
# claude-opus-4-8          $5.00        $25.00       1M
MODELS = {
    "haiku":  "claude-haiku-4-5",
    "sonnet": "claude-sonnet-4-6",
    "opus":   "claude-opus-4-8",
}
MODEL = MODELS["sonnet"]  # ← change this to switch model

MAX_TOKENS = 512         # max tokens in response; adjust based on expected reasoning length and cost tolerance
TEMPERATURE = 0.0        # 0.0 = deterministic; raise to ~0.3 to sample variance across REPETITIONS
CALL_DELAY = 1.0         # seconds between API calls
REQUEST_TIMEOUT = 120    # seconds before giving up on a single API call
MAX_RETRIES = 6          # retry attempts on rate-limit / transient errors
RETRY_BASE_SECONDS = 5.0  # exponential backoff base
RETRY_MAX_SECONDS = 90.0  # cap on backoff sleep

REPETITIONS = 1  # number of full runs; increase to measure output variance across runs

# ================================================================
# CHECKPOINT CONFIG
# Saves progress after each record; a run interrupted mid-way can
# be resumed without re-processing completed records.
# Set RESUME_INCOMPLETE = False to always start from scratch.
# ================================================================
CHECKPOINT_DIR = Path("checkpoints")
RESUME_INCOMPLETE = True

In [264]:
# ================================================================
# REASONING TOGGLE
# Keep True during testing — reasoning shows WHY the model decides
# as it does, which is essential for evaluating prompt quality.
# Set to False for production runs once the prompt is validated,
# to reduce token usage.
# ================================================================
INCLUDE_REASONING = True

PB_CATS = ["Crop development", "Strain development", "Ingredient optimisation", "End product formulation", "Texturization methods", "Food safety & quality", "Health & nutrition", "Consumer & market research", "Impact assessments", "Other"]
F_CATS  = ["Feedstocks", "Target molecule selection", "Strain development", "Bioprocess design", "Ingredient optimisation", "End product formulation", "Texturization methods", "Food safety & quality", "Health & nutrition", "Consumer & market research", "Impact assessments", "Other"]
CM_CATS = ["Cell line development", "Cell culture media", "Bioprocess design", "Scaffolding", "End product formulation", "Food safety & quality", "Health & nutrition", "Consumer & market research", "Impact assessments", "Other"]
CC_CATS = ["Crop development", "Cell line development", "Strain development", "Target molecule selection", "Cell culture media", "Feedstocks", "Bioprocess design", "Scaffolding", "Ingredient optimisation", "End product formulation", "Texturization methods", "Food safety & quality", "Health & nutrition", "Consumer & market research", "Impact assessments", "Other"]

PILLAR_CATS = {"PB": PB_CATS, "F": F_CATS, "CM": CM_CATS, "CC": CC_CATS}

import re

def make_schema(cats, include_reasoning):
    """
    Builds a schema with one boolean field per category (true/false, multi-label)
    instead of a single primary/secondary pick, since a grant can belong to more
    than one research category at once. field_map translates the sanitised
    Python-safe field names (e.g. "Health_nutrition") back to the original
    category label (e.g. "Health & nutrition").
    """
    def field_name(cat):
        return re.sub(r"\W+", "_", cat).strip("_")

    field_map = {field_name(cat): cat for cat in cats}
    fields = {fname: (bool, ...) for fname in field_map}
    if include_reasoning:
        fields["reasoning"] = (str, ...)
    Model = create_model("ClassificationSchema", **fields)
    return Model, field_map

ClassificationSchema, CATEGORY_FIELD_MAP = make_schema(PILLAR_CATS[AP_PILLAR], INCLUDE_REASONING)
print(f"Schema built for {AP_PILLAR}: {list(PILLAR_CATS[AP_PILLAR])}")

client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

def classify_publication(title, abstract, system_prompt):
    user_message = f"Title: {title}\n\nAbstract: {abstract}"
    response = client.messages.parse(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        timeout=REQUEST_TIMEOUT,
        system=[
            {
                "type": "text",
                "text": system_prompt,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[
            {"role": "user", "content": user_message}
        ],
        output_format=ClassificationSchema,
    )
    return response.parsed_output

Schema built for CC: ['Crop development', 'Cell line development', 'Strain development', 'Target molecule selection', 'Cell culture media', 'Feedstocks', 'Bioprocess design', 'Scaffolding', 'Ingredient optimisation', 'End product formulation', 'Texturization methods', 'Food safety & quality', 'Health & nutrition', 'Consumer & market research', 'Impact assessments', 'Other']


In [265]:
def is_retryable_error(exc: Exception) -> bool:
    markers = ["503", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "429",
               "TIMEOUT", "TIMED OUT", "READTIMEOUT", "CONNECTTIMEOUT"]
    return any(m in str(exc).upper() for m in markers)

def retry_sleep_seconds(attempt: int) -> float:
    sleep = min(RETRY_MAX_SECONDS, RETRY_BASE_SECONDS * (2 ** attempt))
    jitter = random.uniform(0.0, min(3.0, sleep * 0.2))
    return sleep + jitter


### 7. Error Handling with Retry

In [266]:
def classify_with_error_handling(row, system_prompt):
    pub_id = row["id"]
    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            result = classify_publication(row["title"], row["abstract"], system_prompt)
            if result is None:
                print(f"  Parse failed for {pub_id}: model returned no structured output")
                return {"id": pub_id, "status": "parse_error", "error": "no structured output"}
            output = {f"{k}_LLM": v for k, v in result.model_dump().items()} # rename columns / keys to indicate LLM output
            output["id"] = pub_id
            output["status"] = "ok"
            return output
        except anthropic.APIError as e:
            last_error = e
            if attempt >= MAX_RETRIES:
                break
            if is_retryable_error(e):
                sleep_s = retry_sleep_seconds(attempt)
                print(f"  Retryable error (attempt {attempt + 1}/{MAX_RETRIES}): {e}. Sleeping {sleep_s:.1f}s.")
                time.sleep(sleep_s)
            else:
                break
    print(f"  API error for {pub_id}: {last_error}")
    return {"id": pub_id, "status": "api_error", "error": str(last_error)}


### 8. Checkpoint Helpers

In [267]:
def get_checkpoint_path(run_idx: int) -> Path:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    return CHECKPOINT_DIR / f"run_{run_idx}_checkpoint.json"

def save_checkpoint(run_idx: int, completed_results: list) -> None:
    path = get_checkpoint_path(run_idx)
    payload = {
        "run_idx": run_idx,
        "completed_ids": [r["id"] for r in completed_results],
        "results": completed_results,
    }
    path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

def load_checkpoint(run_idx: int):
    path = get_checkpoint_path(run_idx)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def delete_checkpoint(run_idx: int) -> None:
    path = get_checkpoint_path(run_idx)
    if path.exists():
        path.unlink()


### 9. Run on Test Data

In [268]:
# Standardise DATASET column names once, separately from the LLM-calling loop below,
# so this (and the comparison cell) can be re-run for free without re-hitting the API
# — e.g. after a kernel restart, or when re-analysing results already in memory / a
# checkpoint file. Grants data has no reliable unique id column, so use row position.
DATASET = DATASET.reset_index(drop=True).rename(columns={"Title": "title", "Abstract": "abstract"})
DATASET["id"] = DATASET.index

In [269]:
all_results = []

for rep in range(REPETITIONS):
    run_idx = rep + 1
    print(f"\n{'='*50}\nRun {run_idx} / {REPETITIONS}\n{'='*50}")

    checkpoint = load_checkpoint(run_idx) if RESUME_INCOMPLETE else None
    if checkpoint:
        completed_results = checkpoint["results"]
        completed_ids = set(checkpoint["completed_ids"])
        print(f"  Resuming: {len(completed_ids)} records already processed.")
    else:
        completed_results, completed_ids = [], set()

    remaining = DATASET[~DATASET["id"].isin(completed_ids)]
    total = len(DATASET)

    for _, row in remaining.iterrows():
        n_done = len(completed_results)
        print(f"  [{n_done + 1}/{total}] {row['id']}")
        result = classify_with_error_handling(row, system_prompt)
        result["run"] = run_idx
        completed_results.append(result)
        save_checkpoint(run_idx, completed_results)
        if n_done + 1 < total:
            time.sleep(CALL_DELAY)

    delete_checkpoint(run_idx)
    all_results.extend(completed_results)



Run 1 / 1
  [1/46] 0
  [2/46] 1
  [3/46] 2
  [4/46] 3
  [5/46] 4
  [6/46] 5
  [7/46] 6
  [8/46] 7
  [9/46] 8
  [10/46] 9
  [11/46] 10
  [12/46] 11
  [13/46] 12
  [14/46] 13
  [15/46] 14
  [16/46] 15
  [17/46] 16
  [18/46] 17
  [19/46] 18
  [20/46] 19
  [21/46] 20
  [22/46] 21
  [23/46] 22
  [24/46] 23
  [25/46] 24
  [26/46] 25
  [27/46] 26
  [28/46] 27
  [29/46] 28
  [30/46] 29
  [31/46] 30
  [32/46] 31
  [33/46] 32
  [34/46] 33
  [35/46] 34
  [36/46] 35
  [37/46] 36
  [38/46] 37
  [39/46] 38
  [40/46] 39
  [41/46] 40
  [42/46] 41
  [43/46] 42
  [44/46] 43
  [45/46] 44
  [46/46] 45


In [270]:
results_df = pd.DataFrame(all_results)
print(f"\nCompleted: {len(results_df)} records across {REPETITIONS} run(s)")
print(f"Successful: {(results_df['status'] == 'ok').sum()}")
print(f"Errors: {(results_df['status'] != 'ok').sum()}")
results_df


Completed: 46 records across 1 run(s)
Successful: 46
Errors: 0


,Crop_development_LLM,Cell_line_development_LLM,Strain_development_LLM,Target_molecule_selection_LLM,Cell_culture_media_LLM,Feedstocks_LLM,Bioprocess_design_LLM,Scaffolding_LLM,Ingredient_optimisation_LLM,End_product_formulation_LLM,Texturization_methods_LLM,Food_safety_quality_LLM,Health_nutrition_LLM,Consumer_market_research_LLM,Impact_assessments_LLM,Other_LLM,reasoning_LLM,id,status,run
0,False,True,True,False,True,True,True,True,False,False,False,False,False,False,False,False,This hub grant explicitly funds manufacturing ...,0,ok,1
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,This grant funds a broad review/overview of al...,1,ok,1
2,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,The grant funds in vitro digestion and in vivo...,2,ok,1
3,False,True,True,True,False,True,True,False,True,True,False,False,False,False,False,False,The CERAFIM project covers the full value chai...,3,ok,1
4,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,This grant is focused on understanding how foo...,4,ok,1
5,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,"This grant focuses on political economy, polic...",5,ok,1
6,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,The abstract is too vague to assign specific t...,6,ok,1
7,False,False,True,False,False,False,False,False,False,True,False,False,False,True,False,False,The grant funds screening and selection of mic...,7,ok,1
8,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,"This grant focuses on stakeholder engagement, ...",8,ok,1
9,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,This grant is primarily focused on consumer be...,9,ok,1


In [271]:
# Ground truth only ever uses "Other" when no specific category applies, so build the
# final predicted label the same way: "Other" is suppressed whenever the LLM also flagged
# at least one specific category True alongside it. raw_predicted_categories keeps the
# LLM's unfiltered output and other_clash flags rows where suppression kicked in, so we
# can see how often the model contradicts itself this way (a high rate may mean the
# prompt's "Other" criteria need tightening) without losing the raw signal.
def build_predictions(row):
    true_labels = [orig for field, orig in CATEGORY_FIELD_MAP.items() if row.get(f"{field}_LLM") == True]
    other_clash = "Other" in true_labels and len(true_labels) > 1
    final_labels = [l for l in true_labels if l != "Other"] if other_clash else true_labels
    return pd.Series({
        "raw_predicted_categories": ", ".join(true_labels),
        "predicted_categories": ", ".join(final_labels),
        "other_clash": other_clash,
    })

results_df[["raw_predicted_categories", "predicted_categories", "other_clash"]] = results_df.apply(build_predictions, axis=1)

result_cols = ["id", "run", "raw_predicted_categories", "predicted_categories", "other_clash", "status"]
if INCLUDE_REASONING:
    result_cols.append("reasoning_LLM")

comparison = DATASET[["id", "title", "abstract", "Research category"]].merge(
    results_df[result_cols], on="id", how="left"
)
comparison = comparison.rename(columns={"Research category": "research_category"})

# Ground truth and predictions are both multi-label comma-separated strings;
# compare them as sets so label order doesn't affect the match.
def to_label_set(s):
    if not isinstance(s, str) or not s.strip():
        return set()
    return {"Other" if l.strip() == "other" else l.strip() for l in s.split(",")}

comparison["research_category_set"] = comparison["research_category"].apply(to_label_set)
comparison["predicted_set"] = comparison["predicted_categories"].apply(to_label_set)
comparison["exact_match"] = comparison["research_category_set"] == comparison["predicted_set"]

# Partial-credit metrics per row: even when the full label set doesn't match exactly,
# how much overlap is there between predicted and true labels?
# - precision: of the labels the LLM predicted, what fraction were actually correct?
# - recall:    of the true labels, what fraction did the LLM find?
# - jaccard:   size of the overlap divided by the size of the union of the two label
#              sets (|true ∩ predicted| / |true ∪ predicted|). Unlike precision/recall,
#              it's one symmetric score that penalises both missed and extra labels at
#              once. 1.0 = identical sets, 0.0 = no overlap at all. e.g. true={X,Y},
#              predicted={X,Z} -> 1 shared label out of 3 distinct labels total = 0.33.
def label_prf(row):
    truth = row["research_category_set"]
    pred = row["predicted_set"]
    if not truth and not pred:
        return pd.Series({"row_precision": 1.0, "row_recall": 1.0, "row_jaccard": 1.0})
    tp = len(truth & pred)
    fp = len(pred - truth)
    fn = len(truth - pred)
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    jaccard = tp / len(truth | pred) if (truth | pred) else float("nan")  # |intersection| / |union|
    return pd.Series({"row_precision": precision, "row_recall": recall, "row_jaccard": jaccard})

comparison[["row_precision", "row_recall", "row_jaccard"]] = comparison.apply(label_prf, axis=1)

# Overall metrics
n = len(comparison)
print(f"Exact match accuracy (full label set matches):     {comparison['exact_match'].mean():.0%}  (n={n})")
print(f"Mean row precision (of predicted labels, % correct): {comparison['row_precision'].mean():.0%}")
print(f"Mean row recall (of true labels, % predicted):       {comparison['row_recall'].mean():.0%}")
print(f"Mean row Jaccard similarity (overlap / union):       {comparison['row_jaccard'].mean():.0%}")
print(f"Other-clash rate (Other flagged alongside another category): {comparison['other_clash'].mean():.0%}  ({int(comparison['other_clash'].sum())} of {n})")

# Per-category precision/recall across the multi-label predictions
all_cats = sorted(CATEGORY_FIELD_MAP.values())
cat_rows = []
for cat in all_cats:
    truth_has = comparison["research_category_set"].apply(lambda s: cat in s)
    pred_has  = comparison["predicted_set"].apply(lambda s: cat in s)
    tp = int((truth_has & pred_has).sum())
    fn = int((truth_has & ~pred_has).sum())
    fp = int((~truth_has & pred_has).sum())
    n_true = int(truth_has.sum())
    recall = tp / n_true if n_true else float("nan")
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    cat_rows.append({
        "research_category": cat, "n_true": n_true, "tp": tp, "fp": fp, "fn": fn,
        "recall": recall, "precision": precision,
    })
cat_stats = pd.DataFrame(cat_rows).set_index("research_category")
cat_stats["recall"] = cat_stats["recall"].map(lambda x: f"{x:.0%}" if pd.notna(x) else "-")
cat_stats["precision"] = cat_stats["precision"].map(lambda x: f"{x:.0%}" if pd.notna(x) else "-")
display(cat_stats)

# Detail table
display_cols = ["id", "title", "abstract", "research_category", "raw_predicted_categories", "predicted_categories", "other_clash"]
if INCLUDE_REASONING:
    display_cols.append("reasoning_LLM")
display_cols += ["exact_match", "row_precision", "row_recall", "row_jaccard"]
comparison[display_cols]

Exact match accuracy (full label set matches):     28%  (n=46)
Mean row precision (of predicted labels, % correct): 41%
Mean row recall (of true labels, % predicted):       61%
Mean row Jaccard similarity (overlap / union):       41%
Other-clash rate (Other flagged alongside another category): 33%  (15 of 46)


,n_true,tp,fp,fn,recall,precision
research_category,,,,,,
Bioprocess design,6,5,8,1,83%,38%
Cell culture media,1,1,2,0,100%,33%
Cell line development,1,0,5,1,0%,0%
Consumer & market research,7,7,8,0,100%,47%
Crop development,1,1,1,0,100%,50%
End product formulation,3,1,10,2,33%,9%
Feedstocks,3,3,4,0,100%,43%
Food safety & quality,4,1,2,3,25%,33%
Health & nutrition,5,4,2,1,80%,67%


,id,title,abstract,research_category,raw_predicted_categories,predicted_categories,other_clash,reasoning_LLM,exact_match,row_precision,row_recall,row_jaccard
0,0,University of Bath Cellular Agriculture Manufa...,Imagine being able to manufacture food anywher...,cross-cutting,"Cell line development, Strain development, Cel...","Cell line development, Strain development, Cel...",False,This hub grant explicitly funds manufacturing ...,False,0.000000,0.0,0.000000
1,1,Alternative Protein Sources - Review for the 2...,The PBE is currently preparing a review on dif...,Impact assessments,Other,Other,False,This grant funds a broad review/overview of al...,False,0.000000,0.0,0.000000
2,2,Oxidation and glycation during gastro-intestin...,"In this project, the impact of meat processing...",Health & nutrition,Health & nutrition,Health & nutrition,False,The grant funds in vitro digestion and in vivo...,True,1.000000,1.0,1.000000
3,3,CERAFIM - Cellular Agriculture for Sustainable...,Our current agricultural practices have a sign...,Feedstocks,"Cell line development, Strain development, Tar...","Cell line development, Strain development, Tar...",False,The CERAFIM project covers the full value chai...,False,0.142857,1.0,0.142857
4,4,Meat replacement and systems of edibility in A...,Alternative proteins can replace meat. Given t...,Consumer & market research,"Consumer & market research, Other",Consumer & market research,True,This grant is focused on understanding how foo...,True,1.000000,1.0,1.000000
5,5,The Political Economy of Meat System Transform...,The industrial food system is a significant co...,Consumer & market research,"Consumer & market research, Other",Consumer & market research,True,"This grant focuses on political economy, polic...",True,1.000000,1.0,1.000000
6,6,Cellular food for sustainable production and i...,"- With CellFood, we can, over the next five ye...",Cell line development,"Consumer & market research, Other",Consumer & market research,True,The abstract is too vague to assign specific t...,False,0.000000,0.0,0.000000
7,7,Designed fermentation for off-flavor\n elimin...,Plant-based foods like those made from faba be...,Ingredient optimisation,"Strain development, End product formulation, C...","Strain development, End product formulation, C...",False,The grant funds screening and selection of mic...,False,0.000000,0.0,0.000000
8,8,Towards a bio-based future: joining forces to ...,The transition towards a sustainable bio-based...,Impact assessments,Other,Other,False,"This grant focuses on stakeholder engagement, ...",False,0.000000,0.0,0.000000
9,9,Meat replacement and systems of edibility in A...,Alternative proteins can replace meat. Given t...,Consumer & market research,"Consumer & market research, Other",Consumer & market research,True,This grant is primarily focused on consumer be...,True,1.000000,1.0,1.000000


### 10. Save to Excel for Prompt Debugging
To assess how well the prompt does at getting the LLM to assign scope and pillar, I need to save the comparison data, then manually review what went wrong and adjust the prompt.
None of this will make it into the final workflow.

Order of working:
1. Create a new version folder in the 1_prompt_debugging folder.
2. Copy in the previous prompt. Label it with the new version number. Make updates as required based on step 6.
3. Edit Step 10 output directory (this step) and Step 5 prompt selection and input data.
4. Run the script from steps 5-10.
5. Manually review the results. Includes both metrics and 
6. Write a text document about v1 results and what changes you want to make to the prompt. Repeat from step 1.

In [272]:
save_dir = Path(f"3_research-category/{AP_PILLAR}_{PROMPT_VERSION}")
save_dir.mkdir(parents=True, exist_ok=True)

summary_df = pd.DataFrame([
    {"metric": "exact_match_accuracy", "value": f"{comparison['exact_match'].mean():.0%}",   "n": n},
    {"metric": "mean_row_precision",   "value": f"{comparison['row_precision'].mean():.0%}", "n": n},
    {"metric": "mean_row_recall",      "value": f"{comparison['row_recall'].mean():.0%}",    "n": n},
    {"metric": "mean_row_jaccard",     "value": f"{comparison['row_jaccard'].mean():.0%}",   "n": n},
    {"metric": "other_clash_rate",     "value": f"{comparison['other_clash'].mean():.0%}",   "n": n},
])

out_path = save_dir / f"{AP_PILLAR}_{PROMPT_VERSION}_{MODEL}_results.xlsx"
with pd.ExcelWriter(out_path) as writer:
    comparison[display_cols].to_excel(writer, sheet_name="results",     index=False)
    cat_stats.to_excel(             writer, sheet_name="by_category")
    summary_df.to_excel(            writer, sheet_name="summary",       index=False)

print(f"Saved to {out_path}")

Saved to 3_research-category\CC_v1\CC_v1_claude-sonnet-4-6_results.xlsx


In [ ]:
# Records where the LLM's predicted label set did not exactly match research_category — for re-run with modified prompt
incorrect_ids = comparison.loc[~comparison["exact_match"], "id"]
incorrect_rescat_data = DATASET[DATASET["id"].isin(incorrect_ids)].reset_index(drop=True)
incorrect_rescat_data

In [233]:
# Manually select specific rows by id for quick re-testing (paste ids from the
# comparison/results tables above). Note: the Section 9 prep cell always resets
# "id" to a fresh 0..n-1 range when it runs, so once this subset goes through the
# script again its ids won't match the ones you selected here — use "title" or
# "research_category" to cross-reference back to the original run if needed.
manual_ids = [10,46]  # <- CHANGE THIS to the ids you want to re-test
manual_test_data = DATASET[DATASET["id"].isin(manual_ids)].reset_index(drop=True)
print(f"Selected {len(manual_test_data)} of {len(manual_ids)} requested ids")
manual_test_data

Selected 2 of 2 requested ids


,title,abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,2027,2028,2029,2030,2031,2032,2033,2034,2035,id
0,Sustainable bioprinting techniques to make pro...,Due to disruptive effects of conventional mass...,NaN,airtable 2025,320000,320000,EUR,376187.0,NaN,320000.0,...,106666.6667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10
1,A Revolutionary Tech for Exceptionally Efficie...,Whilst meat demand is expected to double by 20...,Whilst meat demand is expected to double by 20...,Bruna EU 2025,2432030,2432030,EUR,NaN,NaN,2432030.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46
